
#### 04 - TF-IDF Features

##### Purpose

This notebook builds TF-IDF features from the cleaned support-ticket text.

It:

- reads the persisted modeling dataset
- reuses the existing train, validation, and test split assignments
- fits TfidfVectorizer on training text only
- transforms validation and test text without refitting
- inspects vocabulary, IDF values, feature weights, sparse matrices, and unknown words
- validates that all three datasets share the same feature space

No model training is performed here.


##### 1. Architecture

``` text 

nlp_modeling_dataset
        ↓
dataset_split
        ↓
 ┌───────────────┬─────────────────┬───────────────┐
 ↓               ↓                 ↓
Train         Validation           Test
 ↓               ↓                 ↓
fit +            transform         transform
transform        only              only
 ↓               ↓                 ↓
X_train       X_validation       X_test
   \              |               /
    \             |              /
     └──── same TF-IDF feature space ────┘

```


##### 2. What Problem Does TF-IDF Solve?

Bag of Words answers: How many times does each word appear?

For example: "refund refund billing"

might produce:

- refund  → 2
- billing → 1

But raw counts do not tell us whether a word is actually informative.

A word such as: is

might appear in many tickets.

A word such as: refund

may occur in fewer tickets and therefore carry more useful information.

TF-IDF improves Bag of Words by considering both:

- TF  → frequency inside this document
- IDF → rarity across the training corpus

Then:

TF-IDF = TF × IDF


##### 3. Technologies

- Python
- PySpark
- pandas
- NumPy
- scikit-learn
- TfidfVectorizer
- SciPy sparse matrices
- Delta Lake
- Unity Catalog

##### 4. Imports

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)

from src.project_config import (
    MODELING_TABLE,
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
    EXPECTED_CATEGORIES,
    TFIDF_MAX_FEATURES,
    TFIDF_NGRAM_RANGE,
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
)


##### 5. Verify Configuration

In [0]:
print(f"Modeling table : {MODELING_TABLE}")

print(
    "TF-IDF ngram range :",
    TFIDF_NGRAM_RANGE,
)

print(
    "TF-IDF min_df      :",
    TFIDF_MIN_DF,
)

print(
    "TF-IDF max_df      :",
    TFIDF_MAX_DF,
)

print(
    "TF-IDF max features:",
    TFIDF_MAX_FEATURES,
)


##### 6. Load the Modeling Dataset

In [0]:
modeling_df = spark.table(
    MODELING_TABLE
)

In [0]:
display(
    modeling_df.limit(20)
)

In [0]:
modeling_row_count = (
    modeling_df.count()
)

print(
    f"Modeling row count: "
    f"{modeling_row_count:,}"
)

In [0]:
if modeling_row_count == 0:
    raise ValueError(
        f"Modeling table contains no records: "
        f"{MODELING_TABLE}"
    )

##### 7. Validate Required Columns

In [0]:
required_columns = {
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
}

In [0]:
available_columns = set(
    modeling_df.columns
)

missing_columns = (
    required_columns
    - available_columns
)

if missing_columns:
    raise ValueError(
        "Missing required modeling columns: "
        f"{sorted(missing_columns)}"
    )

print(
    "Input schema validation passed."
)

##### 8. Validate Split Values

In [0]:
actual_splits = {
    row[SPLIT_COL]
    for row in (
        modeling_df
        .select(SPLIT_COL)
        .distinct()
        .collect()
    )
}

In [0]:
expected_splits = {
    "train",
    "validation",
    "test",
}

In [0]:
if actual_splits != expected_splits:
    raise ValueError(
        "Dataset split values are invalid.\n"
        f"Expected: {sorted(expected_splits)}\n"
        f"Actual:   {sorted(actual_splits)}"
    )

print(
    "Dataset split validation passed."
)

##### 9. Validate Categories

In [0]:
actual_categories = {
    row[TARGET_COL]
    for row in (
        modeling_df
        .select(TARGET_COL)
        .distinct()
        .collect()
    )
}

In [0]:
expected_categories = set(
    EXPECTED_CATEGORIES
)

In [0]:
if actual_categories != expected_categories:
    raise ValueError(
        "Dataset categories do not match "
        "project configuration.\n"
        f"Expected: {sorted(expected_categories)}\n"
        f"Actual:   {sorted(actual_categories)}"
    )

print(
    "Category validation passed."
)

##### 10. Create Train / Validation / Test DataFrames

In [0]:
train_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "train"
    )
)

validation_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "validation"
    )
)

test_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "test"
    )
)

In [0]:
train_count = train_df.count()
validation_count = validation_df.count()
test_count = test_df.count()

print(f"Train rows      : {train_count:,}")
print(f"Validation rows : {validation_count:,}")
print(f"Test rows       : {test_count:,}")

##### 11. Convert Required Columns to pandas

In [0]:
train_pdf = (
    train_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

validation_pdf = (
    validation_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

test_pdf = (
    test_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

In [0]:
print(
    "Train shape      :",
    train_pdf.shape,
)

print(
    "Validation shape :",
    validation_pdf.shape,
)

print(
    "Test shape       :",
    test_pdf.shape,
)

##### 12. TF — Term Frequency

TF measures how often a term appears inside one document.

Example: refund refund billing

Counts:

- refund  = 2
- billing = 1

With scikit-learn's default TF-IDF behavior, TF starts from the raw term count.

So conceptually:

TF(refund) = 2
TF(billing) = 1

##### 13. DF — Document Frequency

DF asks: In how many training documents does this term appear?

Suppose:

- D1: customer refund billing
- D2: customer internet problem
- D3: customer billing problem

Then:

customer → appears in 3 documents
billing  → appears in 2 documents
problem  → appears in 2 documents
refund   → appears in 1 document
internet → appears in 1 document

Therefore:

- DF(customer) = 3
- DF(refund)   = 1

##### 14. IDF — Inverse Document Frequency



IDF gives higher importance to rarer terms.

Scikit-learn's default smoothed formula is:

``` text 
idf(t) = log((1 + N) / (1 + df(t))) + 1
```

where:

N     = number of training documents
df(t) = number of training documents containing term t

So:

``` text 

common across many documents
        ↓
lower IDF

rare across documents
        ↓
higher IDF

```

##### 15. Create the TF-IDF Vectorizer

In [0]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
)


``` text 

lowercase=False
    ↓
text is already lowercase

tokenizer=str.split
    ↓
use whitespace tokenization

preprocessor=None
    ↓
no custom extra preprocessing

token_pattern=None
    ↓
disable default regex tokenizer

ngram_range=(1, 1)
    ↓
use individual words only

min_df=1
    ↓
keep a term if it appears in at least 1 training document

max_df=1.0
    ↓
do not remove terms based on being too common

max_features=None
    ↓
do not cap vocabulary size

```

##### 16. Fit TF-IDF on Training Data Only

In [0]:
X_train_tfidf = (
    tfidf_vectorizer
    .fit_transform(
        train_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

##### 17. Transform Validation Data

In [0]:
X_validation_tfidf = (
    tfidf_vectorizer
    .transform(
        validation_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

##### 18. Transform Test Data

In [0]:
X_test_tfidf = (
    tfidf_vectorizer
    .transform(
        test_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

##### 19. Inspect Matrix Shapes

In [0]:
print(
    "Train TF-IDF shape      :",
    X_train_tfidf.shape,
)

print(
    "Validation TF-IDF shape :",
    X_validation_tfidf.shape,
)

print(
    "Test TF-IDF shape       :",
    X_test_tfidf.shape,
)

In [0]:
if not (
    X_train_tfidf.shape[1]
    == X_validation_tfidf.shape[1]
    == X_test_tfidf.shape[1]
):
    raise ValueError(
        "TF-IDF feature dimensions "
        "do not match across datasets."
    )

print(
    "TF-IDF feature-dimension "
    "validation passed."
)

##### 20. Inspect Vocabulary

In [0]:
feature_names = (
    tfidf_vectorizer
    .get_feature_names_out()
)

In [0]:
feature_names

In [0]:
vocabulary_size = len(
    feature_names
)

print(
    f"TF-IDF vocabulary size: "
    f"{vocabulary_size:,}"
)

In [0]:
feature_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            vocabulary_size
        ),
        "term": feature_names,
    }
)

display(
    feature_df
)

##### 21. Is the TF-IDF Vocabulary the Same as BoW?

With the current settings, it will often be the same because both vectorizers are using:

- training text only
- unigrams
- same tokenizer
- no max-feature limit
- min_df = 1
- max_df = 1.0

The major difference is therefore not necessarily the vocabulary.

It is the feature values.

Bag of Words: refund → 3

TF-IDF: refund → weighted decimal value

##### 22. Inspect IDF Values

Scikit-learn stores learned IDF values in: tfidf_vectorizer.idf_

In [0]:
idf_df = pd.DataFrame(
    {
        "term": feature_names,
        "idf": (
            tfidf_vectorizer
            .idf_
        ),
    }
)

In [0]:
display(
    idf_df
    .sort_values(
        "idf",
        ascending=True,
    )
)

low IDF --> word appears in many training tickets

high IDF --> word appears in fewer training tickets

##### 23. Inspect Most Common Terms by IDF

In [0]:
lowest_idf_df = (
    idf_df
    .sort_values(
        "idf",
        ascending=True,
    )
    .head(15)
)

In [0]:
display(
    lowest_idf_df
)

##### 24. Inspect Rarest Terms by IDF

In [0]:
highest_idf_df = (
    idf_df
    .sort_values(
        "idf",
        ascending=False,
    )
    .head(15)
)

In [0]:
display(
    highest_idf_df
)

##### 25. Inspect One Training Ticket

In [0]:
sample_index = 0

sample_ticket = (
    train_pdf
    .iloc[
        sample_index
    ]
)

In [0]:
print(
    "Ticket ID:"
)

print(
    sample_ticket[
        TICKET_ID_COL
    ]
)

print(
    "\nClean text:"
)

print(
    sample_ticket[
        CLEAN_TEXT_COL
    ]
)

print(
    "\nCategory:"
)

print(
    sample_ticket[
        TARGET_COL
    ]
)

##### 26. Inspect Its TF-IDF Vector

In [0]:
sample_vector = (
    X_train_tfidf[
        sample_index
    ]
)

In [0]:
sample_dense = (
    sample_vector
    .toarray()
    .ravel()
)

In [0]:
sample_dense

In [0]:
non_zero_indices = (
    sample_dense
    .nonzero()[0]
)

In [0]:
sample_tfidf_df = pd.DataFrame(
    {
        "term": (
            feature_names[
                non_zero_indices
            ]
        ),
        "tfidf": (
            sample_dense[
                non_zero_indices
            ]
        ),
    }
)

In [0]:
display(
    sample_tfidf_df
    .sort_values(
        "tfidf",
        ascending=False,
    )
)

##### 27. Why Are TF-IDF Values Decimals?

TF-IDF first produces weighted term values.

Then scikit-learn's TfidfVectorizer applies L2 normalization by default.

That means each document vector is scaled so:

sqrt(
    value1²
  + value2²
  + ...
) = 1

This is why TF-IDF values generally appear as decimals between 0 and 1.

##### 28. Verify L2 Normalization

In [0]:
sample_l2_norm = np.sqrt(
    np.sum(
        sample_dense ** 2
    )
)

In [0]:
print(
    f"Sample vector L2 norm: "
    f"{sample_l2_norm:.6f}"
)

##### 29. Demonstrate TF-IDF with a Small Example

In [0]:
demo_corpus = [
    "customer refund billing",
    "customer internet problem",
    "customer billing problem",
]

In [0]:
demo_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
)

In [0]:
demo_matrix = (
    demo_vectorizer
    .fit_transform(
        demo_corpus
    )
)

In [0]:
demo_features = (
    demo_vectorizer
    .get_feature_names_out()
)

print(
    demo_features
)

In [0]:
demo_idf_df = pd.DataFrame(
    {
        "term": demo_features,
        "idf": (
            demo_vectorizer
            .idf_
        ),
    }
)

In [0]:
display(
    demo_idf_df
    .sort_values("idf")
)

##### 30. Compare BoW Intuition to TF-IDF

Consider:

customer refund billing

Bag of Words:

- customer → 1
- refund   → 1
- billing  → 1

All three counts are equal.

TF-IDF recognizes that: customer - appears in every document and therefore provides less distinguishing information.

While: refund -  appears rarely and therefore receives greater importance before normalization.

That is the core improvement.

##### 31. Inspect Sparse Matrix Representation

In [0]:
print(
    X_train_tfidf
)

BoW
dtype = int64
because values are counts

TF-IDF
dtype = float64
because values are weights

##### 32. Calculate TF-IDF Sparsity

In [0]:
total_cells = (
    X_train_tfidf.shape[0]
    * X_train_tfidf.shape[1]
)

non_zero_values = (
    X_train_tfidf.nnz
)

sparsity = (
    1
    - (
        non_zero_values
        / total_cells
    )
)

In [0]:
print(
    f"Total cells     : "
    f"{total_cells:,}"
)

print(
    f"Non-zero values : "
    f"{non_zero_values:,}"
)

print(
    f"Sparsity        : "
    f"{sparsity:.2%}"
)

##### 33. Unknown Words Are Still Ignored

In [0]:
unknown_demo_text = [
    "internet supercalifragilistic"
]

In [0]:
unknown_demo_vector = (
    tfidf_vectorizer
    .transform(
        unknown_demo_text
    )
)

If: internet is known, it gets a feature value.

If: supercalifragilistic was never seen during training, it is ignored.

So TF-IDF improves word importance, but it does not solve the unknown-vocabulary problem.

##### 34. Find Validation-Only and Test-Only Words

In [0]:
training_vocabulary = set(
    feature_names
)

In [0]:
validation_words = set(
    " ".join(
        validation_pdf[
            CLEAN_TEXT_COL
        ]
    ).split()
)

test_words = set(
    " ".join(
        test_pdf[
            CLEAN_TEXT_COL
        ]
    ).split()
)

In [0]:
validation_only_words = (
    validation_words
    - training_vocabulary
)

test_only_words = (
    test_words
    - training_vocabulary
)

In [0]:
print(
    "Validation words not in "
    "training vocabulary:"
)

print(
    sorted(
        validation_only_words
    )
)

print(
    "\nTest words not in "
    "training vocabulary:"
)

print(
    sorted(
        test_only_words
    )
)

##### 35. Compare BoW and TF-IDF Conceptually

###### Bag of Words
-------------
word count

"refund refund billing"

refund  → 2
billing → 1


###### TF-IDF
------
word count
   ×
corpus rarity
   ↓
weighted value

- refund  → potentially stronger weight
- billing → weight depends on document frequency

##### 36. TF-IDF Still Does Not Understand Meaning

TF-IDF improves importance weighting, but: 

refund and reimbursement remain separate vocabulary terms.

Similarly: wifi and wireless have no built-in semantic relationship.

TF-IDF knows: 

- frequency 
- rarity

but not:

- meaning
- context
- semantic similarity

That limitation later motivates embeddings.


##### 37. Prepare Labels

In [0]:
y_train = (
    train_pdf[
        TARGET_COL
    ]
)

y_validation = (
    validation_pdf[
        TARGET_COL
    ]
)

y_test = (
    test_pdf[
        TARGET_COL
    ]
)

In [0]:
print(
    "X_train:",
    X_train_tfidf.shape,
)

print(
    "y_train:",
    y_train.shape,
)

##### 38. Final Validation

In [0]:
assert train_count > 0
assert validation_count > 0
assert test_count > 0

assert vocabulary_size > 0

assert (
    X_train_tfidf.shape[0]
    == train_count
)

assert (
    X_validation_tfidf.shape[0]
    == validation_count
)

assert (
    X_test_tfidf.shape[0]
    == test_count
)

assert (
    X_train_tfidf.shape[1]
    == X_validation_tfidf.shape[1]
    == X_test_tfidf.shape[1]
)

assert (
    len(
        tfidf_vectorizer.idf_
    )
    == vocabulary_size
)

print(
    "Final TF-IDF validation passed."
)

##### 39. Production Design Decisions

This notebook follows these principles:

- Existing persisted train/validation/test assignments are reused.
- No new split is generated.
- TfidfVectorizer is fitted on training data only.
- Validation and test datasets call only transform().
- Vocabulary and IDF statistics therefore come only from training data.
- All datasets share one consistent feature space.
- Sparse matrices are preserved.
- TF-IDF feature engineering remains separate from classifier training.
- Unknown validation/test words are ignored rather than added to the learned vocabulary.
- Configuration comes from src/project_config.py.

##### Key Learnings

TF-IDF builds on Bag of Words rather than replacing the entire idea.

The progression is:

``` text

Vocabulary
     ↓
Bag of Words
     ↓
word counts
     ↓
TF-IDF
     ↓
weighted word importance

```

The three important components are:

``` text

TF
 ↓
How often does the term occur in this document?

DF
 ↓
How many training documents contain the term?

IDF
 ↓
How rare is the term across training documents?

And:

TF-IDF = TF × IDF

```

In scikit-learn, the resulting document vectors are also L2-normalized by default.

Most importantly:

``` text

TRAIN
 ↓
learn vocabulary
learn document frequency
learn IDF

VALIDATION / TEST
 ↓
reuse everything learned from training

```

This is the same train-only fitting principle we used with Bag of Words and with numerical preprocessing in traditional ML.

##### Conclusion

Notebook 04 advances our representation from:

``` text

Bag of Words
    ↓
word counts

to:

TF-IDF
    ↓
weighted word importance

```

We now have numerical feature matrices suitable for traditional machine-learning classifiers:

- X_train_tfidf
- X_validation_tfidf
- X_test_tfidf
- 
and the corresponding labels:

- y_train
- y_validation
- y_test

No model has been trained yet.That comes next.

##### Next Notebook

##### 05_train_ml_classifier

We will finally connect:

``` text

clean support-ticket text
        ↓
TF-IDF
        ↓
Logistic Regression
        ↓
Billing / Cancellation / Login / Technical

```